# New Model creation

> New model creation from trained model

In [ ]:
#| default_exp model_eval.two_image_model

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pathlib import Path
import tensorflow as tf
from fastcore.all import *
import tf2onnx
import numpy as np
import yaml

In [ ]:
#| export
#if platform.system() == 'Linux':
    #custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
    #sys.path.append(str(custom_lib_path))

In [ ]:
#| export
#from dotenv import load_dotenv
#load_dotenv(dotenv_path="/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/.env")

In [ ]:
#| export
#sys.path.append(os.getenv("CV_TOOLS"))
#sys.path.append(os.getenv("PRIVATE_EASY_PIN_DETECTION"))

In [ ]:
#| exporti
from cv_tools.core import *
import matplotlib as mpl
from new_test.mask_generation_patch_image125 import *

In [ ]:
#| export
def prp_fr_mdl(
                  img:np.ndarray # shape of height, width
    ):
    'Prepare for model prediciton'
    max_ = np.max(img)
    if max_ > 2:
        img = img/255.0
    img = np.expand_dims(img, axis=-1)[None, ...]
    img = img.astype(np.float32)
    return img

In [ ]:
#| export 
def frm_im22im1(
    fn:str
    ):
    return Path(fn).name.replace('image2', 'image1')


In [ ]:
#| export
def dilation_(
    img:tf.Tensor,
    threshold:tf.float32,
              t_:int=3):
    img = img > threshold
    img = tf.cast(img, tf.float32)
    for i in range(t_):
        img = tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=[1,1], padding='same')(img)
    return img

    
    

In [ ]:
#| export
def process_image_im1_im2_j(
    im2:np.ndarray,
    im1:np.ndarray
   )->[Tuple[np.ndarray, np.ndarray]]: # return two image of shape (None, 1152, 1632, 1)
    'process image 1 and image2'
    #if len(im1.shape) <3:
        #im1 = np.expand_dims(im1, axis=-1)
        #print(im1.shape)
    #if len(im2.shape) <3:
        #im2 = np.expand_dims(im2, axis=-1)
        #print(im2.shape)



    im1 = im1.astype(np.float32)/255.0
    im2 = im2.astype(np.float32)/255.0
    p_img = np.stack((im1, im2), axis=-1)[None,...]
    return tf.transpose(p_img, (0,3,1,2))

In [ ]:
mpl.rcParams['image.cmap'] = 'gray'

In [ ]:
#| eval: false
src_mdl_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/ETPD-WAR')
dst_mdl_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/ETPD-WAR')

#m_path = src_mdl_path.ls(file_exts='.h5')[0]
model_name='ETPD-WAR1_03.h5'
two_image_model_name = 'ETPD-WAR1_03.keras'
m_path = Path(src_mdl_path, model_name)
m_path_two_img = Path(src_mdl_path, two_image_model_name)

In [ ]:
#| eval: false
im2_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new_two_images/failed/additional/images').ls()[0]
#im2_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new/failed/additional/images').ls()[0]
im1_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im1_cropped')
im1_name = frm_im22im1(im2_path)
im1_fn = Path(im1_path, im1_name)
im2_img = read_img(im2_path)
im1_img = read_img(im1_fn)


In [ ]:
#| eval: false
mdl = tf.keras.models.load_model(m_path, compile=False)
mdl_tim = tf.keras.models.load_model(m_path_two_img, compile=False)

In [ ]:
#| export
def count_obj(msk,threshold:float=0.5,  min_area=0):
    msk = np.array(msk >=threshold, dtype=np.uint8)

    cntrs = find_contours_binary(msk)
    print(f'Number of objects: {len(cntrs)}')
    areas= np.array([cv2.contourArea(i) for i in cntrs])
    min_area, max_area, avg_area = np.min(areas), np.max(areas), np.mean(areas)
    print(f'Min area: {min_area}, Max area: {max_area}, Avg area: {avg_area}')
    cntrs = list(filter(lambda x: cv2.contourArea(x) > min_area, cntrs))

    print(f'After filtering = Number of objects: {len(cntrs)}')
    return cntrs, min_area, max_area, avg_area


In [ ]:
#| export
def erosion_(msk, t_=3):
    msk = msk.astype(np.float32)
    msk = msk * (-1)
    if len(msk.shape) ==2:
        msk = msk[None, ..., None]
    elif len(msk.shape) ==3:
            msk = msk[None, ...]
    else:
        msk = msk
    for i in range(t_):
        msk = tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=[1,1], padding='same')(msk)
    msk = msk * (-1)
    return msk

In [ ]:
#| eval: false
pred_s_msk =mdl.predict(prp_fr_mdl(im2_img))
pred_t_msk = mdl_tim.predict(process_image_im1_im2_j(im2_img, im1_img))

In [ ]:
#| eval: false
cntrs, mi, ma, av = count_obj(pred_s_msk[0,...,0])
cntrs_t, mi_t, ma_t, av_t = count_obj(pred_t_msk[0,...,0])

In [ ]:
#| eval: false
show_(pred_t_msk[0,...,0])

In [ ]:
#| eval: false
show_(pred_s_msk[0,...,0])

In [ ]:
#| eval: false
im1_img_t = np.array(im1_img/255 > 0.3, dtype=np.float32)
pred_s_msk_b =  np.array(pred_s_msk[0,...,0] >= 0.5, dtype=np.float32)
im1_img_t.shape, pred_s_msk[0,...,0].shape

In [ ]:
#| eval: false
com_img = im1_img_t * pred_s_msk_b

In [ ]:
#| eval: false
#show_(com_img)

In [ ]:
#| eval: false
ero_com = erosion_(com_img, t_=2)

In [ ]:
#| eval: false
dil_f = dilation_(com_img[None, ...,None], 0.5, t_=2)


In [ ]:
#| eval: false
erosion_(dil_f, t_=2)

In [ ]:
#| eval: false
dil_com = dilation_(ero_com, threshold=0.5, t_=2)
show_(dil_com[0,...,0])

In [ ]:
#| eval: false
#show_(ero_com[0,...,0])

In [ ]:
#com_imb_r = (-1)* com_img

In [ ]:
#show_(com_imb_r)

In [ ]:
#dilr_ = dilation_(com_imb_r[None, ...,None], 0.5, 2)

In [ ]:
#dil_n = dilr_[0,...,0] * (-1)
#show_(dil_n)

In [ ]:
#dil_ = dilation_(com_img[None,..., None], 0.5, 3)

In [ ]:
#| export
def dilation_(
    img:tf.Tensor,
    threshold:tf.float32,
              t_:int=3):
    img = tf.cast(img, tf.float32)
    for i in range(t_):
        img = tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=[1,1], padding='same')(img)
    return img


In [ ]:
#show_(dil_[0])

In [ ]:
#dil_r = dilation_(dil_ * (-1), threshold=0.5, t_=3) 

In [ ]:
#show_(dil_r[0] * (-1))

In [ ]:
#show_(im1_img_t)

In [ ]:
#nm_ = Path(im2_path).name
#nm_

In [ ]:
#show_doc(show_poster_from_path)

In [ ]:
#| eval: false
msk_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new/failed/additional/masks')
im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new/failed/additional/images')
im1_path 


msk_fn = Path(msk_path, nm_)

In [ ]:
#| eval: false
show_poster_from_path(
    mask_path=msk_fn, 
    im_path=im_path, 
    text='',
    show_='poster'
)

In [ ]:
#| eval: false
msk_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new_two_images/failed/additional/masks')
im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new_two_images/failed/additional/images')


msk_fn = Path(msk_path, nm_)

In [ ]:
#| eval: false
show_poster_from_path(
    mask_path=msk_fn, 
    im_path=im_path, 
    text='',
    show_='poster'
)

# Two image poster

In [ ]:
#| eval: false
msk_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new/failed/additional/masks')

In [ ]:
#| eval: false
pred_n = np.array(pred_s_msk[0,...,0] >= 0.5, dtype=np.uint8)

In [ ]:
#| eval: false
len(find_contours_binary(pred_n))

In [ ]:
#| eval: false
pred_s_msk[]

In [ ]:
#| eval: false
pred_s_msk[0].shape

In [ ]:
#| eval: false
type(pred_s_msk[0])

In [ ]:
#| eval: false
show_(pred_n)

In [ ]:
#| eval: false
cntrs = find_contours_binary(np.array(pred_s_msk[0], dtype=np.uint8))
len(cntrs)

In [ ]:
#| eval: false
n_objs, min_, max_, avg_a = count_obj(pred_s_msk[0], min_area=0)

In [ ]:
#| export
def frm_nrml_to_threshold_mdl(
     mdl_path:str)->tf.keras.Model:
    'Create a another layer called threshold layer'
    def threshold_layer(x, threshold=0.5):
        return tf.cast(x > threshold, tf.float32)
    mdl = tf.keras.models.load_model(mdl_path, compile=False)
    threshold_layer = tf.keras.layers.Lambda(threshold_layer)
    thresholded_output = threshold_layer(mdl.output)
    new_model = tf.keras.Model(
                 inputs=mdl.input, outputs=thresholded_output)
    return new_model


In [ ]:
#new_model = frm_nrml_to_threshold_mdl(m_path)

In [ ]:
#| export
def frm_nrml_to_two_img_model(
    config:dict,
    CHANNEL:int=2,
    HEIGHT:int=1052,
    WIDTH:int=1632,
    dilation_iteration:int=2,
    save_path:str=None
    ):
    'Create same model with just two input image'
   
    # loading model based on config size
    input_shape = (CHANNEL, HEIGHT, WIDTH)
    add_input = tf.keras.layers.Input( 
                                     shape=input_shape,
                                     name='InputImage'
    )
    x = tf.keras.layers.Permute((2,3,1))(add_input)

    model = load_model(config)
    
    image1, image2 = x[:,:,:,0:1], x[:,:,:,1:2]
    image1 = tf.cast(image1 >0.5, tf.float32)
    for i in range(dilation_iteration):
        image1 = tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=[1,1], padding='same')(image1)
    
    model_out = model(image2)
    model_out = tf.cast(model_out >=0.5, tf.float32)
    output_ = tf.keras.layers.Multiply()([
        model_out, image1])
    output_t = tf.cast(output_ >0.5, tf.float32)
    
    new_model = tf.keras.models.Model(
        inputs=[add_input],
        outputs=[output_t]
    )
    if save_path is not None:
        Path(save_path.mkdir(parents=True, exist_ok=True))
        model_name = Path(config['model']['model_name']).stem
        tf.keras.models.save_model(
            new_model,
            f'{save_path}/{model_name}.keras'
        )
    return new_model

In [ ]:
#| export
def frm_nrml_to_two_img_modelv2(
    config:dict,
    CHANNEL:int=2,
    HEIGHT:int=1052,
    WIDTH:int=1632,
    dilation_iteration_rev:int=2,
    dilation_iteration:int=2,
    save_path:str=None
    ):
    'Create same model with just two input image'
   
    # loading model based on config size
    input_shape = (CHANNEL, HEIGHT, WIDTH)
    add_input = tf.keras.layers.Input( 
                                     shape=input_shape,
                                     name='InputImage'
    )
    x = tf.keras.layers.Permute((2,3,1))(add_input)

    model = load_model(config)
    
    image1, image2 = x[:,:,:,0:1], x[:,:,:,1:2]
    image1 = tf.cast(image1 >0.5, tf.float32)
    
    model_out = model(image2)
    model_out = tf.cast(model_out >=0.5, tf.float32)


    output_ = tf.keras.layers.Multiply()([
        model_out, image1])
    
    #revers_ = tf.keras.layers.Multiply()([-1.0, output_])

    revers_ = tf.keras.layers.Lambda(lambda x: x * -1.0)(output_)

    for i in range(dilation_iteration_rev):
        revers_ = tf.keras.layers.MaxPool2D(
            pool_size=(3, 3), 
            strides=[1,1], padding='same')(revers_)

    #output_ = tf.keras.layers.Add()([-1.0, revers_])
    output_ = tf.keras.layers.Lambda(lambda x: x * -1.0)(revers_)

    for i in range(dilation_iteration):
        output_ = tf.keras.layers.MaxPool2D(
            pool_size=(3, 3), 
            strides=[1,1], padding='same')(output_)

    f_out = tf.keras.layers.Multiply()([model_out, output_]) 
    new_model = tf.keras.models.Model(
        inputs=[add_input],
        outputs=[f_out]
    )
    if save_path is not None:
        Path(save_path).mkdir(parents=True, exist_ok=True)
        model_name = Path(config['model']['model_name']).stem
        tf.keras.models.save_model(
            new_model,
            f'{save_path}/{model_name}.keras'
        )
    return new_model

In [ ]:
#| export
def frm_nrml_to_two_img_modelv3(
    config:dict,
    CHANNEL:int=2,
    HEIGHT:int=1052,
    WIDTH:int=1632,
    dilation_iteration_rev:int=2,
    dilation_iteration:int=2,
    im1_threshold:float=0.3,
    pred_threshold:float=0.5,
    save_path:str=None
    ):
    'Create same model with just two input image'
   
    # loading model based on config size
    input_shape = (CHANNEL, HEIGHT, WIDTH)
    add_input = tf.keras.layers.Input( 
                                     shape=input_shape,
                                     name='InputImage'
    )
    x = tf.keras.layers.Permute((2,3,1))(add_input)

    model = load_model(config)
    
    image1, image2 = x[:,:,:,0:1], x[:,:,:,1:2]
    image1 = tf.cast(image1 >im1_threshold, tf.float32)
    
    model_out = model(image2)
    model_out = tf.cast(model_out >=pred_threshold, tf.float32)


    output_ = tf.keras.layers.Multiply()([
        model_out, image1])
    
    #revers_ = tf.keras.layers.Multiply()([-1.0, output_])

    revers_ = tf.keras.layers.Lambda(lambda x: x * -1.0)(output_)

    for i in range(dilation_iteration_rev):
        revers_ = tf.keras.layers.MaxPool2D(
            pool_size=(3, 3), 
            strides=[1,1], padding='same')(revers_)

    #output_ = tf.keras.layers.Add()([-1.0, revers_])
    output_ = tf.keras.layers.Lambda(lambda x: x * -1.0)(revers_)

    for i in range(dilation_iteration):
        output_ = tf.keras.layers.MaxPool2D(
            pool_size=(3, 3), 
            strides=[1,1], padding='same')(output_)

    f_out = tf.keras.layers.Multiply()([model_out, output_]) 
    new_model = tf.keras.models.Model(
        inputs=[add_input],
        outputs=[f_out]
    )
    if save_path is not None:
        print(f'==========================================')
        print('Model saving is  started')
        Path(save_path).mkdir(parents=True, exist_ok=True)
        model_name = Path(config['model']['model_name']).stem
        tf.keras.models.save_model(
            new_model,
            f'{save_path}/{model_name}.keras'
        )
        print(f'==========================================')
        print('Model saving is  done')
    return new_model

In [ ]:
#| eval: false
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_office_test.yaml')

with open (config_path, 'r') as file:
    config = yaml.safe_load(file)

In [ ]:
#src_mdl_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/ModelV10')
#dst_mdl_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/ModelV10_test_thresh03')

In [ ]:
# | eval: false
new_model=frm_nrml_to_two_img_modelv3(
    config=config,
    CHANNEL=2,
    HEIGHT=1152,
    WIDTH=1632,
    dilation_iteration_rev=2,
    dilation_iteration=5,
    save_path=dst_mdl_path
)

In [ ]:
#| export 
def frm_im22im1(
    fn:str
    ):
    return Path(fn).name.replace('image2', 'image1')


In [ ]:
#| eval: false
im2_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im2_cropped_masks/threshold52_new_two_images/failed/additional/images').ls()[0]
im1_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1708/missing_1708_unzip/main_im1_cropped')
im1_name = frm_im22im1(im2_path)
im1_fn = Path(im1_path, im1_name)
im2_img = read_img(im2_path)
im1_img = read_img(im1_fn)


In [ ]:
#| eval: false
im2_img_mdl = prp_fr_mdl(im2_img)
im1_img_mdl = prp_fr_mdl(im1_img)
#dil_im1 = dilation_(im1_img_mdl, threshold=0.5, t_=3)


In [ ]:
#| eval: false
im2_img_mdl = prp_fr_mdl(im2_img)
im1_img_mdl = prp_fr_mdl(im1_img)
dil_im1 = dilation_(im1_img_mdl, threshold=0.5, t_=3)
threshold = 0.5
mdl = load_model(config)
pred  = mdl.predict(im2_img_mdl)
pred_t = tf.cast(pred > threshold, tf.float32)
last = dil_im1 * pred_t
show_(last[0])

In [ ]:
#| eval: false
prd = two_img_mdl.predict(process_image_im1_im2_j(im1_img, im2_img))

In [ ]:
#| eval: false
mdl_inp = process_image_im1_im2_j(im1=im1_img,im2=im2_img)

In [ ]:
#| eval: false
mdl_inp.shape

In [ ]:
#| eval: false
pred_ = two_img_mdl.predict(mdl_inp)

In [ ]:
#| eval: false
pred_.shape

In [ ]:
#| eval: false
show_(pred_[0])

In [ ]:
#| eval: false
#Resize image2 to match the dimensions of image1
#image2 = cv2.resize(image2, (image1.shape[1], image1.shape[0]))

# Create a blended image with transparency
alpha = 0.1  # Adjust the alpha value as per your requirement
blended = cv2.addWeighted(
    im1_img_mdl[0].astype(np.uint8),
                          alpha, 
                          im2_img_mdl[0].astype(np.uint8), 
                          1 - alpha, 0)
    

In [ ]:
#| export
def convert_to_onnx(
                   mdl_path:str,
    
                   
    ):
    'Convert a mdoel to onnx model'
    
    mdl = tf.keras.models.load_model(mdl_path, compile=False)
    m_stem = Path(mdl_path).stem
    parent = Path(mdl_path).parent
    
    onnx_model, _ = tf2onnx.convert.from_keras(
    mdl, 
    opset=13,
    output_path =f'{parent}/{m_stem}.onnx')
    


# Two image model

In [ ]:
#| eval: false
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_office_test.yaml')

with open (config_path, 'r') as file:
    config = yaml.safe_load(file)

In [ ]:
#| eval: false
two_img_mdl = load_model(config)

## Testing two image model

In [ ]:
#| eval: false
#im1_path = Path(r'/home/ai_sintercra.work/Fail_start20240516_unzip/main_im1_cropped')
#im2_path = Path(r'/home/ai_sintercra.work/Fail_start20240516_unzip/main_im2_cropped')

#im1_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im1_cropped')
#im2_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped')

im2_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/failed/additional/images/VFV4.7.9.5_2024022905593626_ID_00325047042818136312407_In_91_r_1_FRONT_Additional Lead_image2.png')
im2_img = read_img(im2_path)

im1_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im1_cropped')
name_ = Path(im2_path).name
im1_fn_ = name_.replace('image2', 'image1')

im1_fn = Path(im1_path, im1_fn_)

In [ ]:
#| eval: false
im1_img = read_img(im1_fn)
im2_img = read_img(im2_path)

In [ ]:
#| eval: false
im1_name = Path(im1_path, im2_path.name)
print(im1_name)
fn_ = im1_name.name
#fn = Path(im1_name).name
im2_name = fn_.replace('image2', 'image1')
#im2_name = Path(im2_path, im2_name)


In [ ]:
#| export
def process_image_im1_im2_j(
    im2:np.ndarray,
    im1:np.ndarray
   )->[Tuple[np.ndarray, np.ndarray]]: # return two image of shape (None, 1152, 1632, 1)
    'process image 1 and image2'
    #if len(im1.shape) <3:
        #im1 = np.expand_dims(im1, axis=-1)
        #print(im1.shape)
    #if len(im2.shape) <3:
        #im2 = np.expand_dims(im2, axis=-1)
        #print(im2.shape)



    im1 = im1.astype(np.float32)/255.0
    im2 = im2.astype(np.float32)/255.0
    p_img = np.stack((im1, im2), axis=-1)[None,...]
    return tf.transpose(p_img, (0,3,1,2))

In [ ]:
#| export
def dilation_(
    img:tf.Tensor,
    threshold:tf.float32,
              t_:int=3):
    img = img > threshold
    img = tf.cast(img, tf.float32)
    for i in range(t_):
        img = tf.keras.layers.MaxPool2D(pool_size=(3, 3), strides=[1,1], padding='same')(img)
    return img

    
    

# Test whether the image1 Erosion works

In [ ]:
#| eval: false
im1_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im1_cropped')
im2_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped')

m_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/ModelV10/time_20_42_24_val_frGrnd0.9675_epoch_197.h5')
model = tf.keras.models.load_model(f'{m_path}',compile=False)
im1_path.ls()

In [ ]:
#| export
def prp_fr_mdl(
                  img:np.ndarray # shape of height, width
    ):
    'Prepare for model prediciton'
    max_ = np.max(img)
    if max_ > 2:
        img = img/255.0
    img = np.expand_dims(img, axis=-1)[None, ...]
    img = img.astype(np.float32)
    return img

In [ ]:
#| eval: false
s_fn = im1_path.ls()[0]
im1 = read_img(s_fn)
name_ = Path(s_fn).name
im2 = read_img(Path(im2_path, name_.replace('image1', 'image2')))
show_(im1)


In [ ]:
#| export
def parse_args_(
    ):
    'Parse arguments'
    import argparse
    parser = argparse.ArgumentParser(description='Create Two image model from single image model')
    parser.add_argument('--config', type=str, help='Config file path but this is single image model config file')
    parser.add_argument('--channel', type=int, default=2, help='Number of channel')
    parser.add_argument('--height', type=int, default=1052, help='Height of the image')
    parser.add_argument('--width', type=int, default=1632, help='Width of the image')
    parser.add_argument('--dilation_iteration_rev', type=int, default=2, help='Number of dilation iteration for reverse')
    parser.add_argument('--dilation_iteration', type=int, default=2, help='Number of dilation iteration')
    parser.add_argument('--save_path', type=str, help='Save path for the model with full name and extension')
    parser.add_argument('--onnx_convert', action='store_true',  help='Convert to onnx')
    return parser.parse_args()


In [ ]:
#| export
def main_():
    args = parse_args_()
    config_path = Path(args.config)
    # just for my understanding
    # .h5 file is 1  channel image
    #  .keras file is 2 channel imageis 
    with open (config_path, 'r') as file:
        config = yaml.safe_load(file)
    
    model_name = Path(config['model']['model_name']).stem

    
    # this save model converts h5 format  to .keras format
    new_model=frm_nrml_to_two_img_modelv3(
        config=config,
        CHANNEL=args.channel,
        HEIGHT=args.height,
        dilation_iteration_rev=args.dilation_iteration_rev,
        dilation_iteration=args.dilation_iteration,
        save_path=args.save_path
    )
    # Thats why here .keras is used when loading the model
    if args.onnx_convert:
        convert_to_onnx(mdl_path=f'{args.save_path}/{model_name}.keras')
        print('Congrats ... ')
        print('Model is converted to onnx')

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
from pathlib import Path
Path.cwd()

In [ ]:
import nbdev
nbdev.nbdev_export('26_model_evaluation.new_model.ipynb')